# 🎢 Optimizadores en Deep Learning
## De SGD con Momentum a Adam: por qué las redes profundas *aprenden* (o no)

**Laboratorio interactivo — Módulo de Machine Learning / Redes Neuronales**

---

### 🎯 Objetivos de aprendizaje

Al terminar este laboratorio vas a poder:

1. **Visualizar** el descenso de gradiente sobre una superficie de pérdida en 2D y entender *por qué* a veces oscila, se estanca o diverge.
2. Explicar con tus propias palabras qué es el **problema del gradiente que desaparece (vanishing gradient)** y por qué ocurre en redes profundas con activaciones sigmoides.
3. Comparar, con una red neuronal implementada **desde cero con NumPy**, cómo evolucionan los gradientes capa por capa durante el entrenamiento.
4. Entender matemáticamente y visualmente la evolución histórica de los optimizadores:

   $$\text{SGD} \;\rightarrow\; \text{SGD + Momentum} \;\rightarrow\; \text{RMSProp} \;\rightarrow\; \text{Adam}$$

5. Comprobar de forma empírica, sobre un **dataset público real** (Iris — flores), que Adam converge más rápido y de forma más estable que SGD "a secas".

---

### 🧠 La idea en una frase

> Un optimizador es la **regla que decide cuánto y hacia dónde mover los pesos** de la red en cada paso, a partir del gradiente de la pérdida. SGD usa *solo* el gradiente actual; Momentum le agrega "inercia" (memoria de hacia dónde veníamos yendo); Adam además **adapta la velocidad de aprendizaje por cada parámetro**, usando tanto la dirección promedio (momentum) como la magnitud típica del gradiente (algo parecido a RMSProp). Esa adaptación es justamente lo que le permite seguir moviendo los pesos incluso cuando el gradiente crudo es minúsculo — que es exactamente lo que pasa en el problema del *vanishing gradient*.

---

### 🗂️ Contenido

1. Setup e instalación de librerías
2. Parte 1 — Descenso de gradiente interactivo en 2D (SGD vs Momentum vs Adam)
3. Parte 2 — Dataset público: Iris (flores)
4. Parte 3 — Una red neuronal desde cero con NumPy (para ver **todo** por dentro)
5. Parte 4 — El problema del vanishing gradient, en vivo
6. Parte 5 — Momentum y Adam al rescate: comparación empírica
7. Parte 6 — Panel interactivo final: "juega" con los optimizadores
8. Conclusiones y ejercicios propuestos


## 1. 🛠️ Setup

Instalamos (si hace falta) e importamos todo lo necesario. Todo el laboratorio corre con **NumPy puro** para la red neuronal (nada de cajas negras tipo Keras/PyTorch) — así vemos exactamente qué hace cada optimizador.


In [ ]:
# Si algo falta en el entorno de Colab, esta celda lo instala.
!pip install -q ipywidgets matplotlib pandas numpy scikit-learn

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
import ipywidgets as widgets
from ipywidgets import interact, interact_manual, FloatSlider, IntSlider, Dropdown, fixed
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (necesario para proyección 3D)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("✅ Librerías cargadas correctamente. Listos para empezar.")


## 2. Parte 1 — Visualizando el descenso de gradiente en 2D

Antes de meternos con redes neuronales, vamos a **ver con los ojos** qué hace un optimizador. Usamos una función de pérdida sintética en 2 dimensiones que tiene la forma de un **valle alargado y curvo** (parecida a la función de Rosenbrock). Esta forma es *a propósito*: es el mismo tipo de geometría "en forma de cañón" que aparece todo el tiempo en redes neuronales reales, y es la que hace que el SGD puro sufra.

$$f(x, y) = (1-x)^2 + 100\,(y - x^2)^2$$

- El **mínimo global** está en $(1, 1)$, donde $f=0$.
- El valle es muy angosto: moverse "bien" requiere avanzar por una curva estrecha, no en línea recta.


In [ ]:
def rosenbrock(x, y):
    '''Función de pérdida sintética con un valle angosto y curvo.'''
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2


def rosenbrock_grad(x, y):
    '''Gradiente analítico de la función anterior: (df/dx, df/dy).'''
    dfdx = -2 * (1 - x) - 400 * x * (y - x ** 2)
    dfdy = 200 * (y - x ** 2)
    return np.array([dfdx, dfdy])


# Visualicemos el "paisaje" de pérdida como un mapa de contornos
xs = np.linspace(-2, 2, 400)
ys = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(xs, ys)
Z = rosenbrock(X, Y)

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis")
ax.plot(1, 1, marker="*", color="red", markersize=18, label="Mínimo global (1,1)")
ax.set_title("Paisaje de pérdida: un valle angosto y curvo\n(escala log para poder verlo bien)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.colorbar(cs, label="log(1+pérdida)")
plt.show()

print("👉 Fijate en la forma de 'banana': ese valle curvo es el reto para el optimizador.")


### 2.1 Implementamos tres optimizadores desde cero

Los tres reciben el mismo gradiente en cada paso; lo único que cambia es **la regla de actualización**.

| Optimizador | Regla de actualización | Idea clave |
|---|---|---|
| **SGD** | $\theta_{t+1} = \theta_t - \eta \, g_t$ | Solo mira el gradiente *actual*. Miope. |
| **SGD + Momentum** | $v_t = \beta v_{t-1} + (1-\beta) g_t$ <br> $\theta_{t+1} = \theta_t - \eta \, v_t$ | Acumula "inercia": promedia direcciones pasadas para no zigzaguear. |
| **Adam** | $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$ <br> $v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$ <br> $\hat m_t = m_t/(1-\beta_1^t)$, $\hat v_t = v_t/(1-\beta_2^t)$ <br> $\theta_{t+1} = \theta_t - \eta \, \hat m_t / (\sqrt{\hat v_t}+\epsilon)$ | Momentum **+** escala el paso dividiendo por la magnitud típica del gradiente (por parámetro). |


In [ ]:
def run_sgd(grad_fn, start, lr, n_iter):
    theta = np.array(start, dtype=float)
    path = [theta.copy()]
    for _ in range(n_iter):
        g = grad_fn(*theta)
        theta = theta - lr * g
        path.append(theta.copy())
    return np.array(path)


def run_momentum(grad_fn, start, lr, n_iter, beta=0.9):
    theta = np.array(start, dtype=float)
    v = np.zeros_like(theta)
    path = [theta.copy()]
    for _ in range(n_iter):
        g = grad_fn(*theta)
        v = beta * v + (1 - beta) * g
        theta = theta - lr * v
        path.append(theta.copy())
    return np.array(path)


def run_adam(grad_fn, start, lr, n_iter, beta1=0.9, beta2=0.999, eps=1e-8):
    theta = np.array(start, dtype=float)
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)
    path = [theta.copy()]
    for t in range(1, n_iter + 1):
        g = grad_fn(*theta)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(theta.copy())
    return np.array(path)

print("✅ SGD, Momentum y Adam implementados desde cero (nada de librerías 'caja negra').")


In [ ]:
def plot_optimizer_paths(lr_sgd=0.001, lr_mom=0.001, lr_adam=0.1, n_iter=200):
    '''Compara visualmente las trayectorias de SGD, Momentum y Adam
    sobre el mismo paisaje de pérdida, partiendo del mismo punto.'''
    start = (-1.5, 2.0)

    path_sgd = run_sgd(rosenbrock_grad, start, lr_sgd, n_iter)
    path_mom = run_momentum(rosenbrock_grad, start, lr_mom, n_iter)
    path_adam = run_adam(rosenbrock_grad, start, lr_adam, n_iter)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # --- Panel izquierdo: trayectorias sobre el mapa de contornos ---
    ax = axes[0]
    cs = ax.contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.85)
    ax.plot(*path_sgd.T, color="white", lw=2, label=f"SGD (lr={lr_sgd})")
    ax.plot(*path_mom.T, color="orange", lw=2, label=f"Momentum (lr={lr_mom})")
    ax.plot(*path_adam.T, color="red", lw=2, label=f"Adam (lr={lr_adam})")
    ax.plot(1, 1, marker="*", color="cyan", markersize=18, label="Mínimo")
    ax.plot(*start, marker="o", color="black", markersize=8, label="Punto de partida")
    ax.set_title("Trayectorias sobre el paisaje de pérdida")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(loc="upper left", fontsize=8)

    # --- Panel derecho: curva de pérdida vs iteración ---
    ax = axes[1]
    ax.plot([rosenbrock(*p) for p in path_sgd], color="white" if False else "gray", label="SGD")
    ax.plot([rosenbrock(*p) for p in path_mom], color="orange", label="Momentum")
    ax.plot([rosenbrock(*p) for p in path_adam], color="red", label="Adam")
    ax.set_yscale("log")
    ax.set_title("Pérdida vs. iteración (escala log)")
    ax.set_xlabel("Iteración"); ax.set_ylabel("f(x,y)")
    ax.legend()

    plt.tight_layout()
    plt.show()

    print(f"Pérdida final -> SGD: {rosenbrock(*path_sgd[-1]):.4f} | "
          f"Momentum: {rosenbrock(*path_mom[-1]):.4f} | "
          f"Adam: {rosenbrock(*path_adam[-1]):.4f}")


interact(
    plot_optimizer_paths,
    lr_sgd=FloatSlider(value=0.001, min=0.0001, max=0.003, step=0.0001, readout_format=".4f", description="lr SGD"),
    lr_mom=FloatSlider(value=0.001, min=0.0001, max=0.003, step=0.0001, readout_format=".4f", description="lr Momentum"),
    lr_adam=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="lr Adam"),
    n_iter=IntSlider(value=200, min=50, max=1000, step=50, description="Iteraciones"),
);


### 🔎 ¿Qué deberías notar al mover los sliders?

- Con **SGD** puro, si subís demasiado el `lr`, la trayectoria empieza a **zigzaguear** de pared a pared del valle en vez de avanzar por el centro. Si lo bajás demasiado, avanza tan lento que ni siquiera se acerca al mínimo en las iteraciones disponibles.
- **Momentum** suaviza ese zigzagueo: promedia la dirección de los últimos pasos, así que "cancela" las oscilaciones perpendiculares al valle y refuerza el avance en la dirección consistente (a lo largo del valle).
- **Adam** normalmente llega mucho más rápido con learning rates *más grandes y sin ajuste fino*, porque adapta el tamaño de paso automáticamente según la magnitud de cada gradiente — no necesitás "adivinar" un `lr` perfecto como con SGD.


## 3. Parte 2 — Dataset público real: Iris (flores)

Para pasar de una función de juguete a un caso real, usamos el dataset **público** más clásico de todo el machine learning: **Iris**, de R.A. Fisher (1936) — medidas de sépalos y pétalos de tres especies de flores. Lo cargamos directo desde un CSV público en GitHub, sin necesidad de descargar nada a mano.

Para quedarnos con un problema de **clasificación binaria** (y poder seguir usando una sola neurona de salida con sigmoide, como en el resto del laboratorio), nos quedamos con las dos especies más parecidas entre sí — *versicolor* y *virginica* — que son bastante más difíciles de separar que *setosa*.


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

df_full = pd.read_csv(url)
print(f"Dataset completo -> Filas: {df_full.shape[0]} | Columnas: {df_full.shape[1]}")

# Nos quedamos solo con las dos especies más difíciles de separar
df = df_full[df_full["species"].isin(["versicolor", "virginica"])].reset_index(drop=True)
df = df.rename(columns={
    "sepal_length": "largo_sepalo", "sepal_width": "ancho_sepalo",
    "petal_length": "largo_petalo", "petal_width": "ancho_petalo",
    "species": "especie"
})
df["diagnostico"] = (df["especie"] == "virginica").astype(int)  # 0 = versicolor, 1 = virginica

print(f"Subconjunto binario -> Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


In [ ]:
# Pequeño EDA: distribución de la variable objetivo y de una variable clave (largo del pétalo)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

df["diagnostico"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"]
)
axes[0].set_xticklabels(["Versicolor (0)", "Virginica (1)"], rotation=0)
axes[0].set_title("Balance de clases")
axes[0].set_ylabel("Cantidad de flores")

axes[1].hist(df.loc[df.diagnostico == 0, "largo_petalo"], bins=20, alpha=0.6, label="Versicolor")
axes[1].hist(df.loc[df.diagnostico == 1, "largo_petalo"], bins=20, alpha=0.6, label="Virginica")
axes[1].set_title("Largo del pétalo según especie")
axes[1].set_xlabel("Largo del pétalo (cm)"); axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Preprocesamiento simple: normalizamos features (media 0, desvío 1) y separamos train/test
from sklearn.model_selection import train_test_split

X = df[["largo_sepalo", "ancho_sepalo", "largo_petalo", "ancho_petalo"]].values.astype(float)
y = df["diagnostico"].values.astype(float).reshape(-1, 1)

# Normalización estándar (clave para que las sigmoides no se saturen de entrada)
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_norm = (X - X_mean) / X_std

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")


## 4. Parte 3 — Una red neuronal desde cero con NumPy

Construimos un **perceptrón multicapa (MLP)** totalmente manual: forward pass, backward pass (backpropagation) y actualización de pesos, todo en NumPy. Usamos a propósito **activación sigmoide en todas las capas ocultas** — es la elección clásica de los años 90 que hace evidente el problema del vanishing gradient en redes profundas.

Arquitectura: entrada (4 features: largo/ancho de sépalo y pétalo) → 5 capas ocultas de 16 neuronas (sigmoide) → salida (1 neurona, sigmoide, clasificación binaria).


In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)  # estabilidad numérica
    return 1 / (1 + np.exp(-z))


def sigmoid_deriv(a):
    '''Derivada de la sigmoide, expresada en función de su propia salida 'a'.'''
    return a * (1 - a)


class MLPDesdeCero:
    '''Red neuronal densa con activación sigmoide en todas las capas,
    entrenada con el optimizador que le indiquemos (sgd / momentum / adam).'''

    def __init__(self, layer_sizes, seed=42):
        rng = np.random.default_rng(seed)
        self.layer_sizes = layer_sizes
        self.W = [rng.normal(0, 0.5, size=(layer_sizes[i], layer_sizes[i + 1]))
                   for i in range(len(layer_sizes) - 1)]
        self.b = [np.zeros((1, layer_sizes[i + 1])) for i in range(len(layer_sizes) - 1)]

        # Estados del optimizador (se usan según corresponda)
        self.mW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(bb) for bb in self.b]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.vb = [np.zeros_like(bb) for bb in self.b]
        self.t = 0

    def forward(self, X):
        activations = [X]
        a = X
        for W, b in zip(self.W, self.b):
            z = a @ W + b
            a = sigmoid(z)
            activations.append(a)
        return activations

    def backward(self, activations, y):
        '''Backpropagation manual. Devuelve gradientes por capa
        y también la norma del gradiente de cada capa (para diagnosticar
        el vanishing gradient).'''
        n = y.shape[0]
        grads_W = [None] * len(self.W)
        grads_b = [None] * len(self.b)
        grad_norms = [None] * len(self.W)

        y_hat = activations[-1]
        delta = (y_hat - y) * sigmoid_deriv(y_hat)  # pérdida: entropía cruzada binaria + sigmoide

        for l in reversed(range(len(self.W))):
            a_prev = activations[l]
            grads_W[l] = (a_prev.T @ delta) / n
            grads_b[l] = delta.mean(axis=0, keepdims=True)
            grad_norms[l] = np.linalg.norm(grads_W[l])
            if l > 0:
                delta = (delta @ self.W[l].T) * sigmoid_deriv(activations[l])

        return grads_W, grads_b, grad_norms

    def update(self, grads_W, grads_b, lr, optimizer="sgd", beta=0.9, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for l in range(len(self.W)):
            gW, gb = grads_W[l], grads_b[l]

            if optimizer == "sgd":
                self.W[l] -= lr * gW
                self.b[l] -= lr * gb

            elif optimizer == "momentum":
                self.mW[l] = beta * self.mW[l] + (1 - beta) * gW
                self.mb[l] = beta * self.mb[l] + (1 - beta) * gb
                self.W[l] -= lr * self.mW[l]
                self.b[l] -= lr * self.mb[l]

            elif optimizer == "adam":
                self.mW[l] = beta1 * self.mW[l] + (1 - beta1) * gW
                self.mb[l] = beta1 * self.mb[l] + (1 - beta1) * gb
                self.vW[l] = beta2 * self.vW[l] + (1 - beta2) * (gW ** 2)
                self.vb[l] = beta2 * self.vb[l] + (1 - beta2) * (gb ** 2)

                mW_hat = self.mW[l] / (1 - beta1 ** self.t)
                mb_hat = self.mb[l] / (1 - beta1 ** self.t)
                vW_hat = self.vW[l] / (1 - beta2 ** self.t)
                vb_hat = self.vb[l] / (1 - beta2 ** self.t)

                self.W[l] -= lr * mW_hat / (np.sqrt(vW_hat) + eps)
                self.b[l] -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
            else:
                raise ValueError(f"Optimizador desconocido: {optimizer}")

    def predict_proba(self, X):
        return self.forward(X)[-1]

    def bce_loss(self, y_hat, y, eps=1e-9):
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))


def entrenar(model, X, y, optimizer="sgd", lr=0.1, epochs=300, **kwargs):
    '''Entrena el modelo y devuelve el historial de pérdida
    y de la norma del gradiente en la PRIMERA capa oculta (la más
    afectada por el vanishing gradient) y en la ÚLTIMA.'''
    loss_history = []
    grad_first_layer = []
    grad_last_layer = []

    for epoch in range(epochs):
        activations = model.forward(X)
        loss = model.bce_loss(activations[-1], y)
        grads_W, grads_b, grad_norms = model.backward(activations, y)
        model.update(grads_W, grads_b, lr=lr, optimizer=optimizer, **kwargs)

        loss_history.append(loss)
        grad_first_layer.append(grad_norms[0])
        grad_last_layer.append(grad_norms[-1])

    return {
        "loss": loss_history,
        "grad_first": grad_first_layer,
        "grad_last": grad_last_layer,
    }

print("✅ MLP desde cero listo: forward, backward y 3 reglas de actualización implementadas.")


## 5. Parte 4 — El problema del *vanishing gradient*, en vivo

Entrenamos nuestra red de **5 capas ocultas sigmoides** con **SGD puro** y comparamos la norma del gradiente en la **primera capa** (la más cercana a la entrada) contra la de la **última capa** (la más cercana a la salida).

**Por qué pasa esto:** en backpropagation, el gradiente de una capa temprana se calcula **multiplicando** las derivadas de todas las capas que vienen después (regla de la cadena). La derivada de la sigmoide, $\sigma'(z) = \sigma(z)(1-\sigma(z))$, tiene un valor **máximo de 0.25** y cae a casi 0 cuando la neurona está saturada. Multiplicar 4 o 5 números que son ≤0.25 da un número *muchísimo* más chico — el gradiente literalmente **desaparece** a medida que viaja hacia atrás.


In [ ]:
arquitectura = [X_train.shape[1], 16, 16, 16, 16, 16, 1]  # 5 capas ocultas sigmoide

modelo_sgd = MLPDesdeCero(arquitectura, seed=1)
hist_sgd = entrenar(modelo_sgd, X_train, y_train, optimizer="sgd", lr=0.5, epochs=400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_sgd["loss"], color="steelblue")
axes[0].set_title("Pérdida durante el entrenamiento (SGD)")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("BCE loss")

axes[1].plot(hist_sgd["grad_first"], label="Norma del gradiente — Capa 1 (temprana)", color="crimson")
axes[1].plot(hist_sgd["grad_last"], label="Norma del gradiente — Capa 5 (última)", color="seagreen")
axes[1].set_yscale("log")
axes[1].set_title("Vanishing gradient: gradiente por capa (escala log)")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("‖gradiente‖ (log)")
axes[1].legend()

plt.tight_layout()
plt.show()

ratio = np.mean(hist_sgd["grad_last"][-20:]) / (np.mean(hist_sgd["grad_first"][-20:]) + 1e-12)
print(f"En las últimas 20 épocas, el gradiente de la última capa es en promedio "
      f"{ratio:.1f}x más grande que el de la primera capa.")
print(f"Pérdida final con SGD: {hist_sgd['loss'][-1]:.4f}")


### 🔎 Lectura del gráfico

Fijate que la curva roja (capa 1, temprana) queda **muy por debajo** de la curva verde (capa 5, cerca de la salida), muchas veces en 1 o 2 órdenes de magnitud. Eso significa que, con SGD puro, **las primeras capas casi no actualizan sus pesos** — la red "aprende" mayormente en las últimas capas y las primeras quedan casi congeladas en su inicialización aleatoria. Esto es exactamente el problema histórico que estancó el entrenamiento de redes profundas hasta mediados de los 2010s.


## 6. Parte 5 — Momentum y Adam al rescate

Repetimos el **mismo experimento**, misma arquitectura, misma inicialización, mismos datos — cambiando **solo el optimizador**. La pregunta: ¿ayuda a que las primeras capas reciban una señal de actualización más fuerte y consistente?


In [ ]:
modelo_mom = MLPDesdeCero(arquitectura, seed=1)
hist_mom = entrenar(modelo_mom, X_train, y_train, optimizer="momentum", lr=0.5, epochs=400, beta=0.9)

modelo_adam = MLPDesdeCero(arquitectura, seed=1)
hist_adam = entrenar(modelo_adam, X_train, y_train, optimizer="adam", lr=0.01, epochs=400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_sgd["loss"], label="SGD", color="gray")
axes[0].plot(hist_mom["loss"], label="SGD + Momentum", color="orange")
axes[0].plot(hist_adam["loss"], label="Adam", color="crimson")
axes[0].set_title("Pérdida durante el entrenamiento")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("BCE loss")
axes[0].legend()

axes[1].plot(hist_sgd["grad_first"], label="SGD — capa 1", color="gray", linestyle="--")
axes[1].plot(hist_mom["grad_first"], label="Momentum — capa 1", color="orange", linestyle="--")
axes[1].plot(hist_adam["grad_first"], label="Adam — capa 1", color="crimson", linestyle="--")
axes[1].set_yscale("log")
axes[1].set_title("Norma del gradiente en la capa 1 (log)")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("‖gradiente‖ (log)")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Pérdida final:")
print(f"  SGD:      {hist_sgd['loss'][-1]:.4f}")
print(f"  Momentum: {hist_mom['loss'][-1]:.4f}")
print(f"  Adam:     {hist_adam['loss'][-1]:.4f}")


### 🔎 Por qué Adam ayuda (y qué NO soluciona)

- **Momentum** acumula gradientes pasados, así que aunque el gradiente crudo de la capa 1 sea diminuto en una época puntual, la "memoria" acumulada de épocas anteriores sigue empujando en una dirección consistente. Esto **atenúa** el problema pero no lo elimina si el gradiente es sistemáticamente casi cero.
- **Adam** va un paso más allá: divide la actualización por $\sqrt{\hat v_t}$, una estimación de la magnitud típica reciente del gradiente **de ese parámetro en particular**. Si un parámetro tiene gradientes consistentemente pequeños, $\hat v_t$ también es pequeño, y el cociente $\hat m_t/\sqrt{\hat v_t}$ se **normaliza hacia un tamaño de paso razonable** — en la práctica, Adam le da a cada parámetro "su propio" learning rate efectivo.
- **Importante — esto NO es magia:** Adam **mitiga** el síntoma (el paso efectivo no colapsa a cero tan rápido), pero **no cambia la causa raíz** (la saturación de la sigmoide). Por eso, en la práctica moderna, el vanishing gradient se ataca con una combinación de: activaciones **ReLU** (derivada no se satura para $z>0$), **inicialización de pesos** adecuada (Xavier/He), **normalización por capas/batch**, **conexiones residuales**, *y* optimizadores adaptativos como Adam. Adam es una pieza del rompecabezas, no la solución completa.


## 7. Parte 6 — Panel interactivo: jugá vos con los optimizadores

Elegí el optimizador, el learning rate y la cantidad de épocas, y volvé a entrenar la red desde cero para ver el efecto inmediatamente.


In [ ]:
def experimento_interactivo(optimizer="adam", lr=0.01, epochs=300, n_capas_ocultas=5):
    arquitectura_exp = [X_train.shape[1]] + [16] * n_capas_ocultas + [1]
    modelo = MLPDesdeCero(arquitectura_exp, seed=1)
    hist = entrenar(modelo, X_train, y_train, optimizer=optimizer, lr=lr, epochs=epochs)

    # Accuracy en test
    y_pred_test = (modelo.predict_proba(X_test) > 0.5).astype(int)
    acc = (y_pred_test == y_test).mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(hist["loss"], color="crimson")
    axes[0].set_title(f"Pérdida — optimizer={optimizer}, lr={lr}, capas ocultas={n_capas_ocultas}")
    axes[0].set_xlabel("Época"); axes[0].set_ylabel("BCE loss")

    axes[1].plot(hist["grad_first"], label="Capa 1 (temprana)", color="crimson")
    axes[1].plot(hist["grad_last"], label="Última capa", color="seagreen")
    axes[1].set_yscale("log")
    axes[1].set_title("Norma del gradiente por capa (log)")
    axes[1].set_xlabel("Época"); axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"Accuracy en test: {acc:.2%} | Pérdida final: {hist['loss'][-1]:.4f}")


interact_manual(
    experimento_interactivo,
    optimizer=Dropdown(options=["sgd", "momentum", "adam"], value="adam", description="Optimizador"),
    lr=FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001, readout_format=".3f", description="Learning rate"),
    epochs=IntSlider(value=300, min=50, max=1000, step=50, description="Épocas"),
    n_capas_ocultas=IntSlider(value=5, min=1, max=8, step=1, description="# capas ocultas"),
);


**Sugerencias para experimentar:**

- Poné `optimizer="sgd"` con `lr=0.01` y subí `n_capas_ocultas` a 8: vas a ver cómo la pérdida casi no baja — el gradiente se "pierde" antes de llegar a la primera capa.
- Con `optimizer="adam"` podés usar el mismo número de capas y, generalmente, vas a ver una convergencia mucho más rápida y estable.
- Probá `optimizer="sgd"` con un `lr` muy alto (por ej. 2.0): vas a ver que la pérdida se dispara (diverge) — SGD no tiene forma de "frenar" un paso demasiado grande, mientras que Adam es más robusto a esto gracias a su normalización adaptativa.


## 8. 📌 Conclusiones

1. **El descenso de gradiente puro (SGD)** funciona, pero es sensible al valle/geometría de la pérdida y al learning rate: muy alto y oscila o diverge; muy bajo y es lentísimo.
2. **Momentum** agrega memoria de la dirección de avance, suavizando oscilaciones y acelerando la convergencia en valles angostos.
3. **El vanishing gradient** ocurre porque backpropagation multiplica derivadas de activación (≤0.25 para la sigmoide) capa por capa; en redes profundas, ese producto colapsa a casi cero antes de llegar a las primeras capas.
4. **Adam** combina momentum (primer momento) con una normalización adaptativa por parámetro basada en la magnitud histórica del gradiente (segundo momento), lo que le permite seguir dando pasos razonables incluso cuando el gradiente crudo es diminuto — por eso se convirtió en el **optimizador por defecto de facto** en la industria.
5. Adam **mitiga** el síntoma del vanishing gradient pero no reemplaza las otras soluciones estructurales: activaciones ReLU/variantes, inicialización He/Xavier, batch/layer normalization y conexiones residuales (ResNets).

### 📝 Ejercicios propuestos

1. Modificá la clase `MLPDesdeCero` para soportar activación **ReLU** en las capas ocultas. Repetí el experimento de la Parte 4/5 con SGD y compará la norma del gradiente en la capa 1 contra la versión sigmoide.
2. Implementá **RMSProp** (Adam sin el término de momentum, solo con la normalización adaptativa) y agregalo al panel interactivo. ¿Se parece más a Momentum o a Adam en sus resultados?
3. Probá el mismo experimento con un dataset propio (por ejemplo, el de riesgo crediticio German Credit Risk que ya usaste en el proyecto grupal) y compará cómo cambian las conclusiones con datos tabulares distintos.
4. Aumentá `n_capas_ocultas` a 15-20 con la activación sigmoide y `optimizer="adam"`. ¿Adam sigue "salvando" el entrenamiento, o llega un punto en que el vanishing gradient es demasiado severo incluso para Adam?

---

*Laboratorio construido con NumPy puro, sin frameworks de deep learning, para que cada paso del optimizador quede completamente visible y auditable.*
